In [2]:
import os
import re
import subprocess
import sys
from pathlib import Path
import pandas as pd
from Bio.PDB import PDBParser, MMCIFParser

### 对设计结果与通过预测的结构进行dockq打分，但是忽略dockq的值，因为fnat不准

In [5]:
with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/dataset/PepSet_dimer/PDB.list', 'r') as f:
    pdbs = [pdb.strip() for pdb in f.readlines()]


with open('dockq.list', 'w') as f:
    for pdb in pdbs:
        # 读取非L链的链名称
        ref_path = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark/dataset/PepSet_AF3_noC_pass/{pdb}.pdb"
        # with open(ref_path, 'r') as pdb_file:
        #     lines = pdb_file.readlines()
        #     chain_ids = set()
        #     for line in lines:
        #         if line.startswith('ATOM') or line.startswith('HETATM'):
        #             chain_id = line[21].strip()
        #             if chain_id != 'L':
        #                 chain_ids.add(chain_id)
        #     if len(chain_ids) == 0:
        #         print(f"Warning: No non-L chains found in {pdb}.pdb")
        #         continue
        #     elif len(chain_ids) > 1:
        #         print(f"Warning: Multiple non-L chains found in {pdb}.pdb: {chain_ids}. Using the first one.")
        #     non_L_chain = sorted(chain_ids)[0]  # 获取第一个非L链的名称
        pred_path = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/inputs'
        pdb_path_list = [file.split(".")[0] for file in os.listdir(pred_path) if file.startswith(f'{pdb}')]
        for file in sorted(pdb_path_list):
            for seed in ['seed-42', 'seed-43', 'seed-44', 'seed-45', 'seed-46']:
                for i in range(5):
                    pred_path = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/outputs/{file}/{seed}_sample-{i}/{file}_{seed}_sample-{i}_model.cif"
                    dockq_cmd = f'DockQ {pred_path} {ref_path} --short --allowed_mismatches 20 --mapping AB:AB'    # 注意，采用mapping时，固定的链需要放在前面
                    f.write(dockq_cmd + '\n')



## 分析DockQ的输出out.log与预测结构的多肽链plddt之间的关系

In [ ]:
import pandas as pd
import numpy as np
import json

method = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/metrices"
rows = []
with open(f'./{method}/dockq_results.txt', 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if i % 2 == 1:
        parts = line.strip().split()
        dockq_score = parts[1].strip()
        irmsd = parts[3].strip()
        lrmsd = parts[5].strip()
        fnat = parts[7].strip()
        model = parts[16].split('/')[-2].strip()
        native = parts[20].split('/')[-1].strip()

        pdb = native.replace('.pdb', '')
        id = model.split('_')[-1]
        pred_json_path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/{method}/{pdb}/{model}/{pdb}_{model}_confidences.json'
        pred_summary_json_path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/{method}/{pdb}/{model}/{pdb}_{model}_summary_confidences.json'

        with open(pred_json_path, 'r') as f_json:
            conf_data = json.load(f_json)
        atom_chain_ids = conf_data.get('atom_chain_ids', [])
        atom_plddts = conf_data.get('atom_plddts', [])
        if len(atom_chain_ids) == 0 or len(atom_chain_ids) != len(atom_plddts):
            continue

        b_chain_plddt = [
            float(plddt)
            for chain_id, plddt in zip(atom_chain_ids, atom_plddts)
            if chain_id == 'B'
        ]
        if len(b_chain_plddt) == 0:
            continue
        pep_plddt = round(float(np.mean(b_chain_plddt)), 4)
        

        with open(pred_summary_json_path, 'r') as f_sum_json:
            sum_conf_data = json.load(f_sum_json)
        iptm = sum_conf_data.get('iptm', None)
        ptm = sum_conf_data.get('ptm', None)
        ranking_score = sum_conf_data.get('ranking_score', None)
        pae_min = min(sum_conf_data.get('chain_pair_pae_min', None)[0][1], sum_conf_data.get('chain_pair_pae_min', None)[1][0])    

        
        dict_row = {'model': model, 'dockq_score': dockq_score, 'pep_plddt': pep_plddt, 'ipTM': iptm, 'ptm': ptm, 'ranking_score': ranking_score, 'pae_min': pae_min, 'irmsd': irmsd, 'lrmsd': lrmsd, 'fnat': fnat, 'native': native}
        rows.append(dict_row)

df = pd.DataFrame(rows)
display(df)

df.to_csv(f'./{method}/docking_results.csv', index=False)

,model,dockq_score,pep_plddt,ipTM,ptm,ranking_score,pae_min,irmsd,lrmsd,fnat,native
0,seed-42_sample-1,0.637,80.5229,0.64,0.73,0.76,1.28,2.197,4.119,0.783,1a0n.pdb
1,seed-43_sample-3,0.681,82.6481,0.66,0.76,0.78,1.22,1.763,3.690,0.783,1a0n.pdb
2,seed-44_sample-3,0.708,84.9685,0.71,0.80,0.83,1.11,1.664,3.585,0.826,1a0n.pdb
3,seed-42_sample-0,0.735,84.0354,0.68,0.77,0.80,1.15,1.540,2.955,0.826,1a0n.pdb
4,seed-44_sample-1,0.614,62.3588,0.43,0.84,0.57,2.10,2.398,5.031,0.821,1cqg.pdb
...,...,...,...,...,...,...,...,...,...,...,...
2545,seed-44_sample-0,0.807,80.2426,0.93,0.87,0.96,1.69,1.415,1.861,0.939,6g5g.pdb
2546,seed-42_sample-0,0.800,80.4662,0.93,0.88,0.96,1.67,1.339,2.022,0.898,6g5g.pdb
2547,seed-44_sample-3,0.803,80.2879,0.93,0.87,0.95,1.71,1.382,1.925,0.918,6g5g.pdb
2548,seed-42_sample-1,0.820,80.5347,0.93,0.88,0.96,1.65,1.272,2.159,0.939,6g5g.pdb


### 分析预测结果中DockQ值小于0.23,在0.23-0.49，0.49-0.80以及0.80以上的pdb个数

In [10]:
# 分析预测结果中DockQ值小于0.23,在0.23-0.49，0.49-0.80以及0.80以上的pdb个数
bins = [0, 0.23, 0.49, 0.80, 1.0]
labels = ['<0.23', '0.23-0.49', '0.49-0.80', '>=0.80']
df['dockq_category'] = pd.cut(df['dockq_score'].astype(float), bins=bins, labels=labels, right=False)
category_counts = df['dockq_category'].value_counts().sort_index()
print(category_counts)

dockq_category
<0.23         0
0.23-0.49     0
0.49-0.80     6
>=0.80       85
Name: count, dtype: int64
